# CTAS public research-table quickstart

This notebook verifies the checksum-bound public CTAS research tables and constructs one reproducible triage cohort. It uses only the Python standard library.

**Scientific boundary:** inclusion in CTAS does not confirm a discovery, class, host, or counterpart. The CTAS score is an operational follow-up ordering aid, not a probability, confidence, or measure of scientific importance.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import os
from urllib.parse import urljoin, urlsplit
from urllib.request import Request, urlopen

PUBLIC_ROOT = os.environ.get("CTAS_PUBLIC_ROOT", "https://jackmcguireastro.github.io/").rstrip("/") + "/"
EXPECTED_MANIFEST_SCHEMA = "ctas.research-table-manifest@1.0.0"
STATUS_PATH = "ctas/data/status.json"
EVENTS_PATH = "ctas/data/research/events.csv"

In [ ]:
root_parts = urlsplit(PUBLIC_ROOT)
if root_parts.scheme not in {"http", "https"} or not root_parts.netloc:
    raise ValueError("CTAS_PUBLIC_ROOT must be an HTTP(S) site-root URL")
if root_parts.scheme != "https" and root_parts.hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise ValueError("Non-HTTPS roots are allowed only for a loopback development server")

def public_url(path: str) -> str:
    candidate = urljoin(PUBLIC_ROOT, path)
    parts = urlsplit(candidate)
    if (parts.scheme, parts.netloc) != (root_parts.scheme, root_parts.netloc):
        raise ValueError(f"Refusing cross-origin CTAS artifact path: {path!r}")
    return candidate

def fetch_bytes(path: str) -> bytes:
    request = Request(public_url(path), headers={"User-Agent": "CTAS-public-quickstart/1.0"})
    with urlopen(request, timeout=45) as response:
        return response.read()

def sha256(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()

def load_json(raw: bytes, label: str) -> dict:
    try:
        document = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise ValueError(f"{label} is not valid UTF-8 JSON") from exc
    if not isinstance(document, dict):
        raise ValueError(f"{label} must be a JSON object")
    return document

In [ ]:
status_raw = fetch_bytes(STATUS_PATH)
status = load_json(status_raw, "CTAS status")
research_pointer = (status.get("artifacts") or {}).get("research_tables")
if not isinstance(research_pointer, dict):
    raise RuntimeError("This CTAS release does not publish a research-table manifest pointer")
manifest_path = research_pointer.get("path")
manifest_sha256 = research_pointer.get("sha256")
if not isinstance(manifest_path, str) or not manifest_path.startswith("ctas/data/research/"):
    raise ValueError("The research manifest path is outside the declared public research directory")

manifest_raw = fetch_bytes(manifest_path)
if sha256(manifest_raw) != manifest_sha256:
    raise ValueError("Research manifest SHA-256 does not match the public status document")
manifest = load_json(manifest_raw, "CTAS research manifest")
if manifest.get("schema") != EXPECTED_MANIFEST_SCHEMA:
    raise ValueError(f"Unsupported research manifest schema: {manifest.get('schema')!r}")
if manifest.get("catalog_content_checksum_sha256") != status.get("catalog_content_checksum_sha256"):
    raise ValueError("Status and research manifest belong to different catalog contents")

verified_tables = {}
seen_paths = set()
for table in manifest.get("tables", []):
    path = table.get("path")
    if not isinstance(path, str) or not path.startswith("ctas/data/research/") or path in seen_paths:
        raise ValueError(f"Invalid or duplicate research-table path: {path!r}")
    seen_paths.add(path)
    raw = fetch_bytes(path)
    if len(raw) != int(table.get("bytes", -1)):
        raise ValueError(f"Byte-count mismatch for {path}")
    if sha256(raw) != table.get("sha256"):
        raise ValueError(f"SHA-256 mismatch for {path}")
    if path.endswith(".csv"):
        parsed_rows = list(csv.reader(io.StringIO(raw.decode("utf-8"), newline="")))
        actual_rows = max(0, len(parsed_rows) - 1)
        if actual_rows != int(table.get("row_count", -1)):
            raise ValueError(f"Row-count mismatch for {path}")
    verified_tables[path] = raw

if EVENTS_PATH not in verified_tables:
    raise ValueError(f"The manifest does not contain {EVENTS_PATH}")
release_id = (status.get("static_snapshot_verification") or {}).get("content_release_id", "not reported")
print(f"Verified {len(verified_tables)} public research tables")
print(f"Catalog content SHA-256: {status['catalog_content_checksum_sha256']}")
print(f"Content release ID: {release_id}")

In [ ]:
events_text = verified_tables[EVENTS_PATH].decode("utf-8")
reader = csv.DictReader(io.StringIO(events_text, newline=""))
events = list(reader)
required_columns = {"event_id", "name", "ra_deg", "dec_deg", "ctas_score", "status", "classification"}
missing_columns = required_columns.difference(reader.fieldnames or [])
if missing_columns:
    raise ValueError(f"events.csv is missing required columns: {sorted(missing_columns)}")
if len({row['event_id'] for row in events}) != len(events):
    raise ValueError("events.csv contains duplicate stable event UUIDs")
print(f"Loaded {len(events):,} unique public event rows")

In [ ]:
def finite_float(value: str | None) -> float | None:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None

def coordinate_valid(row: dict[str, str]) -> bool:
    ra = finite_float(row.get("ra_deg"))
    dec = finite_float(row.get("dec_deg"))
    return ra is not None and dec is not None and 0.0 <= ra < 360.0 and -90.0 <= dec <= 90.0

TERMINAL_STATUSES = {"retracted", "bogus"}
SCORE_THRESHOLD = 70.0
cohort = [
    row for row in events
    if coordinate_valid(row)
    and (finite_float(row.get("ctas_score")) or -1.0) >= SCORE_THRESHOLD
    and row.get("status", "").strip().lower() not in TERMINAL_STATUSES
]
cohort.sort(key=lambda row: (-(finite_float(row.get("ctas_score")) or -1.0), row.get("name", ""), row["event_id"]))
print(f"Cohort: {len(cohort):,} coordinate-valid, non-terminal records with CTAS score >= {SCORE_THRESHOLD:g}")
print("This is a reproducible follow-up-ordering cohort, not a scientific-interest or discovery sample.")

In [ ]:
print(f"{'Name':<20} {'UUID prefix':<12} {'Score':>7} {'RA (deg)':>11} {'Dec (deg)':>11}  {'Reported class / label'}")
print("-" * 95)
for row in cohort[:20]:
    name = row.get("name", "")[:20]
    event_id = row.get("event_id", "")[:12]
    score = finite_float(row.get("ctas_score"))
    ra = finite_float(row.get("ra_deg"))
    dec = finite_float(row.get("dec_deg"))
    label = row.get("classification") or "Unclassified"
    print(f"{name:<20} {event_id:<12} {score:7.2f} {ra:11.6f} {dec:11.6f}  {label}")

In [ ]:
cohort_columns = [
    "event_id", "name", "ra_deg", "dec_deg", "ctas_score", "status",
    "classification", "discovery_time", "discovery_survey", "n_observations",
    "n_spectra", "primary_source_url",
]
buffer = io.StringIO(newline="")
writer = csv.DictWriter(buffer, fieldnames=cohort_columns, extrasaction="ignore", lineterminator="\n")
writer.writeheader()
writer.writerows(cohort)
cohort_csv_bytes = buffer.getvalue().encode("utf-8")
print(f"In-memory cohort CSV: {len(cohort_csv_bytes):,} bytes")
print(f"Cohort CSV SHA-256: {sha256(cohort_csv_bytes)}")
print("The variable cohort_csv_bytes is ready to save or pass to another tool explicitly.")

## Next steps

Use stable `event_id` values to open the corresponding CTAS dossiers before interpreting a target. The normalized event table is designed for catalog-scale selection; complete source-native evidence, conflicts, query receipts, limitations, and original-provider links remain in the checksum-bound dossier shards. Record the catalog-content checksum and access time with any derived result.